# Transaction Foundation Model on Ray — Part 6: Downstream fraud — raw vs FM vs fusion

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~10 min

---

This is the payoff of the whole series: does the foundation-model embedding actually help catch fraud, compared to the features a team already has? We answer it with a controlled experiment. At `small`/`full` the embedding table is millions of rows by hundreds of dimensions — too big to pull onto one node and fit in-process — so the classifier itself trains on the cluster: Ray Data streams the embeddings and `XGBoostTrainer` distributes the boosting across workers (CPU here at `mini`, GPUs at `small`/`full`). It's the same `ScalingConfig` knob the pretrain and embed stages turn.


In [ ]:
import sys, os, json, logging

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ray

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts
paths = artifact_paths(get_demo_base_dir(), SCALE)

# How this scale fits the classifier: mini = 1 CPU worker (reproducible, CI-safe);
# small/full = GPU workers. Same code path either way.
ds_cfg = load_scale(SCALE)["downstream"]
NUM_WORKERS, USE_GPU = ds_cfg["num_workers"], ds_cfg["use_gpu"]

# XGBoost reports a metric every boosting round; keep that out of the notebook.
for _n in ("ray.data", "ray.train", "ray.tune"):
    logging.getLogger(_n).setLevel(logging.ERROR)
ray.data.DataContext.get_current().enable_progress_bars = False

## Does the embedding beat the features you already have?

We train the **same XGBoost** on three feature sets, so the only thing that changes is the *representation*:

1. **raw** — the target transaction's own fields: NVIDIA's 13-column baseline (amount, hour, date, MCC, channel/use-chip, merchant, city, state, zip, card identity), ordinal-encoded. This is the strong hand-built baseline a fraud team has today — the FM has to beat *this*, not a toy 4-field version.
2. **fm** — only the FM embedding of the card's history window, with none of the raw fields.
3. **fusion** — the embedding concatenated with the raw fields (Nubank's joint-fusion recipe).

The lift of **fm** and **fusion** over **raw** is the case for a transaction foundation model: it says you can drop or augment a hand-tuned feature pipeline with one pretrained encoder.

The evaluation follows NVIDIA's transaction-FM blueprint so the numbers are comparable: the **temporal 80/10/10 split** from Part 2 (train on the past, test on the most recent 10% — no leakage), per-transaction last-event labels, and **PR-AUC at natural fraud prevalence** (the downsampled normals are importance-weighted back to ~0.1%). An eval fingerprint makes two runs comparable only if they scored the exact same held-out set.

## Fit all three with one identical recipe

`run_downstream` runs the whole comparison on the cluster. It PCA-reduces the 512-dim
embedding to 64 (NVIDIA's choice), ordinal-encodes the raw categoricals (fit on train),
and fits `raw`, `fm`, and `fusion` with **one identical, fully-trained XGBoost recipe**.
Using the same recipe for all three is what makes the comparison fair: the only thing
that differs is the representation, so `fusion` (raw features + embedding) can only match
or beat `raw`. It runs in a GPU worker task and writes the metrics + per-sample test
scores we plot below.


In [ ]:
from src.downstream import run_downstream, print_summary

# One call runs the fair comparison on the cluster: PCA-reduce the 512-dim embedding to
# 64 (NVIDIA's choice), ordinal-encode the raw categoricals, and fit raw / fm / fusion
# with ONE identical, fully-trained XGBoost recipe — so the representation is the only
# variable and `fusion` (raw + embedding) can only match or beat `raw`. The fit runs in
# a GPU worker task and writes metrics + per-sample test scores for the curves below.
summary = run_downstream(
    embeddings_path=paths["embeddings"],
    output_dir=paths["downstream"],
    pca_dim=ds_cfg["pca_dim"],
    use_gpu=USE_GPU,
)
print_summary(summary)


In [ ]:
import pandas as pd

r = summary["results"]
tbl = pd.DataFrame(r).T[["auc_roc", "pr_auc"]].rename(
    columns={"auc_roc": "AUC-ROC", "pr_auc": "PR-AUC"})
print(f"train={summary['n_train']:,} ({summary['train_fraud_rate']:.1%} fraud)  "
      f"test={summary['n_test']:,} ({summary['test_fraud_rate']:.4%} fraud)  "
      f"emb_dim={summary['embedding_dim']}")
display(tbl.style.format("{:.4f}"))
lift = summary["fusion_lift_pr_auc"]
print(f"\nFM-only PR-AUC lift vs raw:  {summary['fm_lift_pr_auc']:+.4f}")
print(f"Fusion  PR-AUC lift vs raw:  {lift:+.4f}   "
      f"({'the FM adds signal' if lift > 0 else 'no lift at this scale'})")


## Reading this honestly at `mini` scale

At `mini` the foundation model trailed the raw baseline — and that's the expected, honest result, not a bug. The encoder here was pretrained for **2 CPU epochs at 64 dimensions, 2 layers**: it has barely learned, so a 64-dim summary of recent history can't yet compete with the target transaction's own raw fields. So `mini` is a smoke test of the *pipeline and the evaluation harness*, not a model you'd ship.

What transfers regardless of scale is the **methodology**: a leakage-free temporal split, metrics reported at the true ~0.1% prevalence (where the published TFMs are measured), and a controlled comparison where representation is the only variable. Run the `small`/`full` GPU pretrain and the picture inverts — `fm` and `fusion` overtake `raw`, matching the lift NVIDIA's blueprint and FATA-Trans report on this dataset. The point of this notebook is the apparatus that will *show* that lift, run honestly at a scale that fits on a CPU.

## The curves

Two views of the same held-out scores, both at natural prevalence. **ROC** shows how well each representation *ranks* fraud above normal — readable here because the `mini` model is weak (at `full` a strong model pushes AUC-ROC toward 1.0, where it saturates and hides differences). **Precision–Recall** shows the operational reality: at ~0.1% fraud, precision collapses, which is exactly why PR-AUC — not accuracy or ROC — is the number a fraud team actually optimizes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

pred = pd.read_parquet(os.path.join(paths["downstream"], "test_predictions.parquet"))
colors = {"raw": "#4C78A8", "fm": "#F58518", "fusion": "#54A24B"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for name in ("raw", "fm", "fusion"):
    d = pred[pred["feature_set"] == name]
    fpr, tpr, _ = roc_curve(d["label"], d["proba"])
    prec, rec, _ = precision_recall_curve(d["label"], d["proba"])
    ax1.plot(fpr, tpr, color=colors[name], label=name, lw=1.6)
    ax2.plot(rec, prec, color=colors[name], label=name, lw=1.6)
ax1.plot([0, 1], [0, 1], "k--", lw=0.7)
ax1.set(title="ROC", xlabel="False positive rate", ylabel="True positive rate")
ax2.set(title="Precision–Recall (the operative view at 0.1% fraud)",
        xlabel="Recall", ylabel="Precision")
ax1.legend(); ax2.legend()
plt.tight_layout(); plt.show()


## Takeaways

- **It scales by config, not by rewrite.** The same `read_parquet → map_batches → XGBoostTrainer` code fits on one CPU worker at `mini` and on `num_workers` GPU workers at `small`/`full` — only `ScalingConfig` changes. Pulling millions of embeddings onto one node and fitting in-process was the bottleneck this stage removes; Ray Data shards the read and the training so the driver never holds the full matrix.
- **The comparison is controlled**: one XGBoost recipe, three feature sets, representation the only variable — so any lift is attributable to the embedding, not to tuning.
- **The protocol is honest and comparable**: temporal split (no leakage), per-transaction labels, PR-AUC at the true ~0.1% prevalence, and an eval fingerprint so two runs are only compared when they scored the same held-out set. It matches NVIDIA's transaction-FM blueprint. (Distributed boosting partitions the data across workers, so exact metric values are reproducible for a given worker count, not bit-identical across counts.)
- **At `mini` the FM trails raw** — a 2-epoch, 64-dim CPU encoder is undertrained. The value of this stage is the apparatus; the lift shows up with the `small`/`full` GPU pretrain, where pretrained TFMs beat hand-built features on this benchmark.

---

## Next

**Part 7 — Online serving with Ray Serve**: deploy the encoder behind an HTTP endpoint that returns an embedding and a fraud score in one call, caching the static card-level fields and autoscaling replicas under load.
